In [ ]:
!pip install -q xgboost pytest
import os, sys, shutil, importlib, json, time
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path("/content/drive/MyDrive/KP-TA MCI")
FIGSHARE_DIR = DRIVE / "02. Figshare"
W2, W3 = DRIVE / "05. Week 2", DRIVE / "06. Week 3"
HASIL = W3 / "hasil"; HASIL.mkdir(parents=True, exist_ok=True)

# salin cache ke disk lokal supaya baca cepat
LOCAL = Path("/content/cache")
for nm in ("fitur_li_v2", "pose_grid32"):
    if not (LOCAL / nm).exists():
        shutil.copytree(W2 / nm, LOCAL / nm)
FITUR_DIR, POSE_GRID_DIR = LOCAL / "fitur_li_v2", LOCAL / "pose_grid32"

REPO = "/content/fog-benchamark-starter"
if not os.path.isdir(REPO):
    !git clone -q --depth 1 https://github.com/bungagana/fog-benchamark-starter {REPO}
for f in ("li_kinematik_xgb.py", "stgcn.py"):
    shutil.copy(W3 / "kode_model" / f, f"{REPO}/models/{f}")
Path(REPO, "config", "paths.local.yaml").write_text(f'figshare: "{FIGSHARE_DIR}"\n', encoding="utf-8")
os.environ["FOG_FITUR_DIR"] = str(FITUR_DIR)
os.environ["FOG_POSE_DIR"] = str(POSE_GRID_DIR)
if REPO not in sys.path: sys.path.insert(0, REPO)

from fogbench import data as D, splits, schema
from fogbench.threshold import pick_threshold
from fogbench.metrics import sample_metrics
from fogbench.evaluate import protocol
import torch
recs = D.load("figshare")
print("Rekaman:", len(recs), "| pasien:", len({r.subject for r in recs}))
print("fitur:", len(list(FITUR_DIR.glob("PDFE*.npz"))), "| pose:", len(list(POSE_GRID_DIR.glob("*.npz"))))
print("GPU:", torch.cuda.is_available())

In [ ]:
!cd {REPO} && python -m pytest tests -q 2>&1 | tail -3
!cd {REPO} && python scripts/selftest.py 2>&1 | tail -3

In [ ]:
OUT_LI1 = HASIL / "li_1seed"
!cd {REPO} && python scripts/run_experiment.py --model models.li_kinematik_xgb --dataset figshare --seeds 0 --out "{OUT_LI1}"
!cd {REPO} && python scripts/validate_results.py "{OUT_LI1}" --n-fold 35

In [ ]:
OUT_LI3 = HASIL / "li_3seed"
!cd {REPO} && python scripts/run_experiment.py --model models.li_kinematik_xgb --dataset figshare --out "{OUT_LI3}"
!cd {REPO} && python scripts/validate_results.py "{OUT_LI3}" --n-fold 35

In [ ]:
def satu_fold(model_factory, recs, subjek_uji, out_dir, model_name, dataset="figshare", seeds=None, fs_eval=None):
    p = protocol()
    seeds = list(seeds or p["seeds"]); fs_eval = int(fs_eval or p["fs_evaluasi_hz"])
    fold = next(f for f in splits.loso_folds({r.subject for r in recs}) if f.test_subject == subjek_uji)
    train, test = splits.split_recordings(recs, fold)
    p_seeds, t_train = [], 0.0
    for sd in seeds:
        model = model_factory(seed=sd)
        t0 = time.perf_counter(); model.fit(train); t_train += time.perf_counter() - t0
        p_seeds.append(np.concatenate([D.to_eval_rate(model.predict_proba(r), r.fs, fs_eval) for r in test]))
    p_mean = np.mean(p_seeds, axis=0)
    y_eval = np.concatenate([D.to_eval_rate(r.y, r.fs, fs_eval, is_label=True) for r in test])
    n = min(len(p_mean), len(y_eval)); p_mean, y_eval = p_mean[:n], y_eval[:n]; p_seeds = [q[:n] for q in p_seeds]
    model_t = model_factory(seed=seeds[0]); model_t.fit(train)
    y_tr = np.concatenate([D.to_eval_rate(r.y, r.fs, fs_eval, is_label=True) for r in train])
    p_tr = np.concatenate([D.to_eval_rate(model_t.predict_proba(r), r.fs, fs_eval) for r in train])
    m = min(len(y_tr), len(p_tr)); thr = pick_threshold(y_tr[:m], p_tr[:m])
    t0 = time.perf_counter(); _ = model_t.predict_proba(test[0]); infer_ms = (time.perf_counter() - t0) * 1000
    path = schema.write_fold(Path(out_dir) / f"fold_{fold.index:02d}.json", model=model_name, dataset=dataset,
                             fold=fold.index, test_subject=fold.test_subject, train_subjects=list(fold.train_subjects),
                             fs_eval=fs_eval, threshold=thr, y_true=y_eval, p_mean=p_mean, p_per_seed=p_seeds,
                             seeds=seeds, train_time_s=t_train, infer_time_ms=infer_ms, model_card=model_t.card())
    return path

def jalankan_loso(factory, nama, out_dir, seeds):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    folds = splits.loso_folds({r.subject for r in recs})
    for f in folds:
        if (out_dir / f"fold_{f.index:02d}.json").exists():
            continue
        t0 = time.time()
        satu_fold(factory, recs, f.test_subject, out_dir, nama, seeds=seeds)
        print(f"fold {f.index:02d}/{len(folds)} pasien {f.test_subject} | {time.time()-t0:.0f} s", flush=True)
print("siap")

In [ ]:
import models.li_kinematik_xgb as ML; importlib.reload(ML)
T = HASIL / "uji_kesetaraan"; T.mkdir(exist_ok=True)
satu_fold(ML.factory, recs, 4, T, "li-kinematik-xgb", seeds=[0])
a = json.loads((T / "fold_04.json").read_text())
b = json.loads((OUT_LI1 / "fold_04.json").read_text())
print("p_mean sama:", np.allclose(a["p_mean"], b["p_mean"], atol=1e-6), "| ambang sama:", np.isclose(a["threshold"], b["threshold"]))

In [ ]:
os.environ["FOG_STGCN_SKEMA"] = "pinggul"
os.environ["FOG_STGCN_EPOCHS"] = "30"
import models.stgcn as S; importlib.reload(S)
OUT_ST = HASIL / "stgcn_pinggul_1seed"
jalankan_loso(S.factory, "stgcn-pinggul", OUT_ST, seeds=[0])
!cd {REPO} && python scripts/validate_results.py "{OUT_ST}" --n-fold 35

In [ ]:
exec(open(W3 / "kode_model" / "analisis_w3.py").read())

In [ ]:
a = json.loads((HASIL/"uji_kesetaraan"/"fold_04.json").read_text())
b = json.loads((HASIL/"li_1seed"/"fold_04.json").read_text())
print(np.allclose(a["p_mean"], b["p_mean"], atol=1e-6), np.isclose(a["threshold"], b["threshold"]))

In [ ]:
src = (W3 / "kode_model" / "stgcn.py").read_text()
assert "DROPOUT" in src, "stgcn.py di Drive belum versi patch"
os.environ["FOG_STGCN_EPOCHS"] = "30"
import models.stgcn as S; importlib.reload(S)
import subprocess
ANTREAN = [("skala",  [0, 1, 2], "stgcn_skala_3seed"),
           ("piksel", [0, 1, 2], "stgcn_piksel_3seed")]
for skema, seeds, folder in ANTREAN:
    os.environ["FOG_STGCN_SKEMA"] = skema
    jalankan_loso(S.factory, f"stgcn-{skema}", HASIL / folder, seeds)
    r = subprocess.run(["python", "scripts/validate_results.py", str(HASIL / folder), "--n-fold", "35"],
                       cwd=REPO, capture_output=True, text=True)
    print(folder, "->", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:])